In [ ]:
import pandas as pd
import os
import numpy as np
import h5py
import scclone2dr
df = pd.read_csv('/data/users/04_share_reanalysis_results/02_melanoma/MEL_CNN_abundances_no_plate_effect_correction.csv')

# Showing the bias induced by the pipetting effect

In [ ]:
COHORTS = ['melanoma','aml']
gene_set_collections = ['geneOncoKB','gene','c6','hallmarks', 'c2_pid']
cohort2clonemodes = {'melanoma': ["scatrex",'scatrex_rawcounts_scvi', 'phenograph'], 'aml':['phenograph']}

# melanoma cohort
COHORT = COHORTS[0]
gene_set_collection = gene_set_collections[3]
clonemode = cohort2clonemodes[COHORT][0]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
concentration_DMSO = '100'
concentration_drug = '5'
path_info_cohort = None

datamodule = scclone2dr.data.RealData(path_fastdrug=path_fastdrug, path_rna=path_rna)
data_ref = datamodule.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

In [ ]:
sample2control_wellpos = datamodule.FD.sample2control_wellpos
sample_names = datamodule.sample_names

In [ ]:
sample2nu_ratio = {}
for i, sample in enumerate(sample_names):
    sample2nu_ratio[sample] = {}
    n0 = data_ref["n_rna"][0,i]
    ntumor = torch.sum(data_ref["n_rna"][1:,i])
    w0 = n0 / (n0 + ntumor)
    wtumor = ntumor / (n0 + ntumor)
    n0well = data_ref['n0_c'][:,i]
    ntotalwell = data_ref['n_c'][:,i]
    ntumorwell = ntotalwell - n0well
    for wellID in range(len(n0well)):
        if ntotalwell[wellID] > 0:
            sample2nu_ratio[sample][wellID] =  (ntumorwell[wellID] / n0well[wellID]) * (w0 / wtumor)

In [ ]:
nugrid = np.zeros((16,len(sample_names)))
for idsample, sample in enumerate(sample_names):
    wellpos = sample2control_wellpos[sample]
    for i in range(wellpos.shape[0]):
        nugrid[wellpos[i][0]-1,idsample] = sample2nu_ratio[sample][i]
b = plt.imshow(nugrid)
plt.colorbar(b)
plt.xlabel('Patient')
plt.ylabel('Well horizontal coordinate')
plt.show()


nugrid = np.zeros((24,len(sample_names)))
for idsample, sample in enumerate(sample_names):
    wellpos = sample2control_wellpos[sample]
    for i in range(wellpos.shape[0]):
        nugrid[wellpos[i][1]-1,idsample] = sample2nu_ratio[sample][i]
b = plt.imshow(nugrid)
plt.colorbar(b)
plt.xlabel('Patient')
plt.ylabel('Well vertical coordinate')
plt.show()


In [ ]:
from scipy.stats import ttest_rel

odd_means = []
even_means = []

for idsample, sample in enumerate(sample_names):
    wellpos = sample2control_wellpos[sample]
    values = sample2nu_ratio[sample]
    
    odd_vals = [values[i] for i in range(len(values)) if wellpos[i][1] % 2 == 1]
    even_vals = [values[i] for i in range(len(values)) if wellpos[i][1] % 2 == 0]
    
    odd_means.append(np.mean(odd_vals))
    even_means.append(np.mean(even_vals))

print(ttest_rel(odd_means, even_means))

In [ ]:
nugrid = np.zeros((16,24))
counts = np.zeros((16,24))
for idsample, sample in enumerate(sample_names):
    wellpos = sample2control_wellpos[sample]
    values = sample2nu_ratio[sample]
    

    for i in range(wellpos.shape[0]):
        row = wellpos[i][0]-1
        col = wellpos[i][1]-1
        
        nugrid[row,col] += values[i]
        counts[row,col] += 1
nugrid /= counts
nugrid = np.nan_to_num(nugrid)
    
plt.imshow(nugrid, aspect='auto')
plt.colorbar()
plt.xlabel('Patient')
plt.ylabel('Pipetting order')
plt.title('Value ordered by pipetting sequence')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_positions = 24  # adjust
mean_by_position = np.zeros(n_positions)
count_by_position = np.zeros(n_positions)

for sample in sample_names:
    wellpos = sample2control_wellpos[sample]
    values = list(sample2nu_ratio[sample].values())
    
    # center values by patient
    values_centered = values - np.mean(values)
    
    for i in range(len(values)):
        col = wellpos[i][1] - 1
        mean_by_position[col] += values_centered[i]
        count_by_position[col] += 1

mean_by_position /= count_by_position

plt.plot(range(1, n_positions+1), mean_by_position, 'o-')
plt.xlabel('Column coordinate of the well on plate', fontsize=14)
plt.ylabel(r'Average of patient-centered $\nu$-ratio', fontsize=14)
#plt.savefig('pipetting_effect.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import spearmanr
import numpy as np

x = np.arange(1, n_positions+1)  # dispense positions = w
y = mean_by_position              # patient-centered nu_ratio = y

rho, pval = spearmanr(x, y)

print(f"Spearman correlation rho = {rho:.3f}, p-value = {pval:.4f}")

# What was the typical number of drugs tested for melanoma (with how many replicates at each concentration) and for aml

In [ ]:
import pandas as pd
import os
import numpy as np
import h5py
df = pd.read_csv('/data/users/04_share_reanalysis_results/02_melanoma/MEL_CNN_abundances_no_plate_effect_correction.csv')
df = pd.read_csv('/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv')

In [ ]:
concentrations = np.unique(df['Concentration'].values)
sample_names = np.unique(df['SampleID'].values)

In [ ]:
import pandas as pd
import numpy as np

#sample_names = ["MACEGEJ"]#["MYHAJIS"]

for concentration in concentrations:
    print('Concentration', concentration)
    
    per_sample_n_drugs = []
    per_sample_counts = []

    for sample in sample_names:
        df_temp = df[
            (df['Concentration'] == concentration) &
            (df['SampleID'] == sample)
        ].copy()

        if df_temp.empty:
            # No data → treat as zero
            per_sample_n_drugs.append(0)
            per_sample_counts.append(pd.Series(dtype=float))
            continue

        df_temp['Drug'] = df_temp['Drug'].str.lower()

        # number of unique drugs in this sample
        per_sample_n_drugs.append(df_temp['Drug'].nunique())

        # counts per drug
        counts = df_temp.groupby("Drug").size()
        per_sample_counts.append(counts)

    # ---- Mean & STD of number of drugs tested ----
    mean_n_drugs = np.mean(per_sample_n_drugs)
    std_n_drugs = np.std(per_sample_n_drugs, ddof=1) if len(per_sample_n_drugs) > 1 else 0

    # ---- Mean & STD per drug across samples ----
    if per_sample_counts:
        counts_df = pd.concat(per_sample_counts, axis=1).fillna(0)
        mean_per_drug = counts_df.mean(axis=1)
        max_per_drug = counts_df.max(axis=1)
        std_per_drug = counts_df.std(axis=1, ddof=1).fillna(0)
        
        mean_per_drug = mean_per_drug.sort_values(ascending=False)
        max_per_drug = max_per_drug.sort_values(ascending=False)
        std_per_drug = std_per_drug.loc[mean_per_drug.index]
    else:
        mean_per_drug = pd.Series(dtype=float)
        max_per_drug = pd.Series(dtype=float)
        std_per_drug = pd.Series(dtype=float)

    print(f"Mean number of drugs tested (over all patients): {mean_n_drugs:.2f}")
    #print(f"STD number of drugs tested: {std_n_drugs:.2f}")
    print("\nMean number of replicates (over all patients) per drug:")
    print(max_per_drug)
    #print("\nSTD counts per drug:")
    #print(std_per_drug)
    print("------------------------------------------------------")
    print("\n")